In [ ]:
# Imports
import os
from pathlib import Path
markers = (".git", "Program")
current_dir = Path.cwd()
project_root = next((path for path in (current_dir, *current_dir.parents) if any((path / marker).exists() for marker in markers)), current_dir)
os.chdir(project_root)

from helper_functions import *
import itertools
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from plot import *
from arch import arch_model
import re
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf, pacf, adfuller, coint, grangercausalitytests
from technicals import *
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = True
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HSI", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the current date
current_date = modify_current_date(start, index_name)

In [ ]:
def granger_causality_test(stocks, dfs, max_lag=20):
    """
    Perform Granger causality test and cross-correlation analysis.

    Parameters:
    - stocks (list): List of two stock symbols.
    - dfs (list): List of two DataFrames containing stock data.
    - max_lag (int): Maximum lag to test.

    Returns:
    - valid_data (pd.DataFrame): DataFrame with aligned percent changes.
    - gc_1to2 (dict): Granger causality results for stock_1 to stock_2.
    - gc_2to1 (dict): Granger causality results for stock_2 to stock_1.
    """

    stock_1, stock_2 = stocks
    stock_1_df, stock_2_df = dfs

    for df in (stock_1_df, stock_2_df):
        if "Percent Change" not in df.columns:
            df["Percent Change"] = df["Close"].pct_change()

    common_dates = stock_1_df.index.intersection(stock_2_df.index)
    valid_data = pd.DataFrame({
        stock_1: stock_1_df.loc[common_dates, "Percent Change"],
        stock_2: stock_2_df.loc[common_dates, "Percent Change"]
    }).dropna()

    try:
        gc_1to2 = grangercausalitytests(valid_data[[stock_2, stock_1]], maxlag=max_lag, verbose=False)
        gc_2to1 = grangercausalitytests(valid_data[[stock_1, stock_2]], maxlag=max_lag, verbose=False)
    except ValueError as e:
        match = re.search(r"Maximum allowable lag is (\d+)", str(e))
        if match:
            max_lag = int(match.group(1))
            print(f"Adjusting max_lag to {max_lag}.")
            gc_1to2 = grangercausalitytests(valid_data[[stock_2, stock_1]], maxlag=max_lag, verbose=False)
            gc_2to1 = grangercausalitytests(valid_data[[stock_1, stock_2]], maxlag=max_lag, verbose=False)
        else:
            raise

    gc_results = pd.DataFrame({
        f"{stock_1} → {stock_2}": [gc_1to2[lag][0]["ssr_ftest"][1] for lag in range(1, max_lag + 1)],
        f"{stock_2} → {stock_1}": [gc_2to1[lag][0]["ssr_ftest"][1] for lag in range(1, max_lag + 1)]
    }, index=range(1, max_lag + 1))

    try:
        display(gc_results)
    except NameError:
        print(gc_results)

    return valid_data, gc_1to2, gc_2to1

In [ ]:
# Perform Granger causality test and cross-correlation analysis
stocks = ["^GSPC", "PLTR"]
dfs = [get_df(stock, current_date, max_period=True) for stock in stocks]
valid_data, gc_1to2, gc_2to1 = granger_causality_test(stocks, dfs, max_lag=20)

In [ ]:
stocks = stock_market("2019-01-01", current_date, "^GSPC", False, bloomberg=True)

In [ ]:
stocks_miss = []
for stock in tqdm(stocks):
    try:
        df = get_df(stock, current_date)
        df = df[df.index >= "2020-01-01"]
        recent_close = df["Close"].iloc[-1]
    except Exception as e:
        print(f"Error fetching data for {stock}: {e}")
        stocks_miss.append(stock)
        continue